In [1]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import netCDF4 as n
import glob
import xarray as xr
import seaborn as sns
import matplotlib.lines as mlines
import pandas as pd
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.stats import linregress
from collections import Counter
sns.set_context("paper")
sns.set_style("whitegrid")
computed_name = 'ComputedScalarsHelene'


In [2]:
mnt_pth = '/Volumes/FMac/computing/data/antarctica/' #mount path
pth_ismip6hack= mnt_pth + 'ismip6/hackathon/ComputedScalars/' #write folder
pth_ismip6 =mnt_pth + 'ismip6/hackathon/' #read folder
pth_helene =mnt_pth + 'ismip6/hackathon/ComputedScalarsHelene/'

expnames = ['expAE02', 'expAE03', 'expAE04', 'expAE05']
removeFileID = ['IMAU_UFEMISM1', 'IMAU_UFEMISM2', 'IMAU_UFEMISM3', 'IMAU_UFEMISM4',
               'DOE_MALI_4km', 'DOE_MALI_8km_Ant95', 'DOE_MALI_8km_AntMean'] # specify models to remove
 
metadata_file = 'Metadata.txt'
df_time = pd.read_csv('/Users/fmcc0003/issm/projects/ISMIP6Hackathon/GroupBMBtodSLR/data_analysis/tables/time_maximum_dSLR.csv')
# Generate loop_info DataFrame

In [3]:
def get_model(string,exp):
    return string.split('/')[-1].split('AIS_')[-1].split('_'+str(exp))[0]

In [4]:
models=[]
paths =[]
exps =[]
region_names = ['Amundsen','Ross','FilchnerRonne']
dic_sectors ={}
dic_sectors ['Amundsen']=np.array([4])
dic_sectors ['Ross']=np.array([2,6])
dic_sectors ['FilchnerRonne']=np.array([5,11])
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
df['AIS']=0
for j in range(1,4):
    df['region_'+str(j)] =0
for key in dic_sectors.keys():
    df[key] = 0
    
df_n = df.copy()
factor = -1* 365.25 * 24 * 3600/900

In [5]:
for i in range (df.shape[0]):
    #load the datasets:
    pth_way = df.iloc[i].Path
    pth_dslc = f'{pth_ismip6hack}/{df.iloc[i].Experiment}/dslc_anom/computed_dslc_anom_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
    pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/iareafl/computed_iareafl_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
    
    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
    d2 = xr.open_dataset(pth_area,decode_times=False )
    # dSLC
    d3 = xr.open_dataset(pth_dslc,decode_times=False )
    
    ###### start with AIS
    t_max = df_time.iloc[i].AIS
    tmin = np.where(d.time.values==t_max)[0][0]
 
    # melting per area
    x = np.cumsum(d.shelfmelt.values[:tmin]*factor/d2.iareafl.values[:tmin]) #cumulative bmb
    y = np.cumsum(-1*d3.dslc_anom.values[:tmin]) #cumulative dSLC

    if len(x)>1:
        slope, intercept, r_value, p_value, std_err = linregress(x, y)
        df['AIS'].iloc[i]= slope
        df_n['AIS'].iloc[i]= intercept   
        
    else:
        df['AIS'].iloc[i]= np.nan
        df_n['AIS'].iloc[i]= np.nan
    
    #all the sectors:
    for j in range(1,19):
        t_max = df_time.iloc[i]['sector_'+str(j)]
    
        tmin = np.where( d.time.values==t_max)[0][0]
 
        x = np.cumsum(d['shelfmelt_sector_'+str(j)].values[:tmin]*factor/d2['iareafl_sector_'+str(j)].values[:tmin]) #GT/m^3 yr
        y = np.cumsum(-1*d3['dslc_anom_sector_'+str(j)].values[:tmin])
        mnan1 = np.isnan(y)
        mnan2 = np.isnan(x)
        mnan = mnan1 | mnan2
    
        if len(x[~mnan])>1:
            slope, intercept, r_value, p_value, std_err = linregress(x[~mnan], y[~mnan])
            df['sector_'+str(j)].iloc[i] = slope
            df_n['sector_'+str(j)].iloc[i] = intercept
        
        else:
            df['sector_'+str(j)].iloc[i] = np.nan
            df_n['sector_'+str(j)].iloc[i] = np.nan
            
    #all the regions
    for j in range(1,4):
        t_max = df_time.iloc[i]['region_'+str(j)]
    
        tmin =np.where( d.time.values==t_max)[0][0]
        x = np.cumsum(d['shelfmelt_region_'+str(j)].values[:tmin]* factor/d2['iareafl_region_'+str(j)].values[:tmin])  #GT/m^3 yr
        y = np.cumsum(-1*d3['dslc_anom_region_'+str(j)].values[:tmin])

        mnan1 = np.isnan(y)
        mnan2 = np.isnan(x)
        mnan = mnan1 | mnan2

        if len(x[~mnan])>1:
            slope, intercept, r_value, p_value, std_err = linregress(x[~mnan], y[~mnan])
            df['region_'+str(j)].iloc[i] = slope
            df_n['region_'+str(j)].iloc[i] = intercept
        
        else:
            df['region_'+str(j)].iloc[i] = np.nan
            df_n['region_'+str(j)].iloc[i] = np.nan
            
    #specific region,combination of sectors: Amundsen, Ross, Fris
    for key in dic_sectors.keys():

        t_max = 2300
        for j in dic_sectors[key]:
            t_max1 = df_time.iloc[i]['sector_'+str(j)]
            t_max = np.min([t_max,t_max1])
    
        tmin = np.where(d.time.values==t_max)[0][0]
        x = np.cumsum((d['shelfmelt_sector_'+str(j)].values[:tmin].copy() * factor/d2['iareafl_sector_'+str(j)].values[:tmin].copy()), axis=0) #cumulative melt
        y = np.cumsum((-1*d3['dslc_anom_sector_'+str(j)].values[:tmin].copy()), axis = 0) # gives cumulative dSLC line
        mnan1 = np.isnan(x)
        mnan2 = np.isnan(y)
        mnan = mnan1 | mnan2

        if len(x[~mnan])>1:
            print(f'some nans!{i} at {df.iloc[i].Model}')
            slope, intercept, r_value, p_value, std_err = linregress(x[~mnan], y[~mnan])

            df[key].iloc[i] = slope
            df_n[key].iloc[i] = intercept
                
        else:
            print('pnli nans',df.iloc[i].Model)
            df[key].iloc[i] = np.nan
            df_n[key].iloc[i] =np.nan

df.to_csv('/Users/fmcc0003/issm/projects/ISMIP6Hackathon/GroupBMBtodSLR/data_analysis/tables/linear_dslc_sensitivity_from_BMB_max.csv', index=False)
df_n.to_csv('/Users/fmcc0003/issm/projects/ISMIP6Hackathon/GroupBMBtodSLR/data_analysis/tables/linear_dslc_sensitivity_from_BMB_max_intercept.csv', index=False)

/var/folders/v3/r3qsjbkx2gl_9r5s_n0726wxcg0fz6/T/ipykernel_19026/476237660.py:24: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['AIS'].iloc[i]= slope
/var/folders/v3/r3qsjbkx2gl_9r5s_n0726wxcg0fz6/T/ipykernel_19026/476237660.py:24: Settin

some nans!0 at DC_ISSM
some nans!0 at DC_ISSM
some nans!0 at DC_ISSM
some nans!1 at IGE_ElmerIce
some nans!1 at IGE_ElmerIce
some nans!1 at IGE_ElmerIce
some nans!2 at ILTS_SICOPOLIS
some nans!2 at ILTS_SICOPOLIS
some nans!2 at ILTS_SICOPOLIS
some nans!3 at LSCE_GRISLI2
some nans!3 at LSCE_GRISLI2
some nans!3 at LSCE_GRISLI2
some nans!4 at LSCE_GRISLI
some nans!4 at LSCE_GRISLI
some nans!4 at LSCE_GRISLI
some nans!5 at NCAR_CISM1
some nans!5 at NCAR_CISM1
some nans!5 at NCAR_CISM1
some nans!6 at NCAR_CISM2
some nans!6 at NCAR_CISM2
some nans!6 at NCAR_CISM2
some nans!7 at NORCE_CISM2-MAR364-ERA-t1
some nans!7 at NORCE_CISM2-MAR364-ERA-t1
some nans!7 at NORCE_CISM2-MAR364-ERA-t1
some nans!8 at NORCE_CISM3-MAR364-ERA-t1-local
some nans!8 at NORCE_CISM3-MAR364-ERA-t1-local
some nans!8 at NORCE_CISM3-MAR364-ERA-t1-local
some nans!9 at NORCE_CISM3-MAR364-ERA-t1-nonlocal
some nans!9 at NORCE_CISM3-MAR364-ERA-t1-nonlocal
some nans!9 at NORCE_CISM3-MAR364-ERA-t1-nonlocal
some nans!10 at NORCE_

/var/folders/v3/r3qsjbkx2gl_9r5s_n0726wxcg0fz6/T/ipykernel_19026/476237660.py:49: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['sector_'+str(j)].iloc[i] = np.nan
/var/folders/v3/r3qsjbkx2gl_9r5s_n0726wxcg0fz6/T/ipykernel_19026/476237660.

some nans!22 at ULB_fETISh-KoriBU2
some nans!22 at ULB_fETISh-KoriBU2
some nans!22 at ULB_fETISh-KoriBU2
some nans!23 at UNN_Ua
some nans!23 at UNN_Ua
some nans!23 at UNN_Ua
some nans!24 at UTAS_ElmerIce
some nans!24 at UTAS_ElmerIce
some nans!24 at UTAS_ElmerIce
some nans!25 at VUB_AISMPALEO
some nans!25 at VUB_AISMPALEO
some nans!25 at VUB_AISMPALEO
some nans!26 at VUW_PISM1
some nans!26 at VUW_PISM1
some nans!26 at VUW_PISM1
some nans!27 at VUW_PISM1_s1
some nans!27 at VUW_PISM1_s1
some nans!27 at VUW_PISM1_s1
some nans!28 at VUW_PISM1_s2
some nans!28 at VUW_PISM1_s2
some nans!28 at VUW_PISM1_s2
some nans!29 at VUW_PISM1_s3
some nans!29 at VUW_PISM1_s3
some nans!29 at VUW_PISM1_s3
some nans!30 at VUW_PISM1_s4
some nans!30 at VUW_PISM1_s4
some nans!30 at VUW_PISM1_s4
some nans!31 at VUW_PISM2
some nans!31 at VUW_PISM2
some nans!31 at VUW_PISM2
some nans!32 at VUW_PISM2_s1
some nans!32 at VUW_PISM2_s1
some nans!32 at VUW_PISM2_s1
some nans!33 at VUW_PISM2_s2
some nans!33 at VUW_PISM2_

/var/folders/v3/r3qsjbkx2gl_9r5s_n0726wxcg0fz6/T/ipykernel_19026/476237660.py:57: RuntimeWarning: invalid value encountered in divide
  x = np.cumsum(d['shelfmelt_region_'+str(j)].values[:tmin]* factor/d2['iareafl_region_'+str(j)].values[:tmin])  #GT/m^3 yr
/var/folders/v3/r3qsjbkx2gl_9r5s_n0726wxcg0fz6/T/ipykernel_19026/476237660.py:82: RuntimeWarning: invalid value encountered in divide
  x = np.cumsum((d['shelfmelt_sector_'+str(j)].values[:tmin].copy() * factor/d2['iareafl_sector_'+str(j)].values[:tmin].copy()), axis=0) #cumulative melt


some nans!44 at NORCE_CISM3-MAR364-ERA-t1-local
some nans!44 at NORCE_CISM3-MAR364-ERA-t1-local
some nans!44 at NORCE_CISM3-MAR364-ERA-t1-local
some nans!45 at NORCE_CISM3-MAR364-ERA-t1-nonlocal
some nans!45 at NORCE_CISM3-MAR364-ERA-t1-nonlocal
some nans!45 at NORCE_CISM3-MAR364-ERA-t1-nonlocal
some nans!46 at NORCE_CISM3-MAR364-ERA-t1
some nans!46 at NORCE_CISM3-MAR364-ERA-t1
some nans!46 at NORCE_CISM3-MAR364-ERA-t1
some nans!47 at NORCE_CISM4-MAR364-ERA-t1-local
some nans!47 at NORCE_CISM4-MAR364-ERA-t1-local
some nans!47 at NORCE_CISM4-MAR364-ERA-t1-local
some nans!48 at NORCE_CISM4-MAR364-ERA-t1-nonlocal
some nans!48 at NORCE_CISM4-MAR364-ERA-t1-nonlocal
some nans!48 at NORCE_CISM4-MAR364-ERA-t1-nonlocal
some nans!49 at NORCE_CISM4-MAR364-ERA-t1
some nans!49 at NORCE_CISM4-MAR364-ERA-t1
some nans!49 at NORCE_CISM4-MAR364-ERA-t1
some nans!50 at NORCE_CISM4-MAR364-JRA-t1
some nans!50 at NORCE_CISM4-MAR364-JRA-t1
some nans!50 at NORCE_CISM4-MAR364-JRA-t1
some nans!51 at NORCE_CISM5-

In [6]:
#######plot the scatter too
dic_area_of_interest ={}
dic_area_of_interest['AIS']= ['']
dic_area_of_interest['WestAntarctica'] = ['_region_1']
dic_area_of_interest['EastAntarctica'] = ['_region_2']
dic_area_of_interest['Peninsula'] = ['_region_3']
dic_area_of_interest['Amundsen'] = ['_sector_4']
dic_area_of_interest['FilchnerRonne'] = ['_sector_5','_sector_11']
dic_area_of_interest['Ross'] = ['_sector_2','_sector_6']

In [7]:
expnames = ['expAE02', 'expAE03', 'expAE04', 'expAE05']
removeFileID = ['IMAU_UFEMISM1', 'IMAU_UFEMISM2', 'IMAU_UFEMISM3', 'IMAU_UFEMISM4',
               'DOE_MALI_4km', 'DOE_MALI_8km_Ant95', 'DOE_MALI_8km_AntMean'] # specify models to remove
 
metadata_file = 'Metadata.txt'
def get_model(string,exp):
    return string.split('/')[-1].split('AIS_')[-1].split('_'+str(exp))[0]

# Generate loop_info DataFrame
models= []
paths = []
exps = []
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
df['AIS']=0
df_org = df.copy()

In [8]:
metadata_file = '/Users/fmcc0003/issm/projects/ISMIP6Hackathon/GroupTFtoOcean/data_analysis/Metadata_icefront.txt'
df_info = pd.read_csv(metadata_file)                 
df_info_model = df_info[df_info['fileID']==df.iloc[i].Model ]
df_info_model_exp = df_info_model[df_info_model['Experiment']==exp]
df_dslcs = pd.read_csv('/Users/fmcc0003/issm/projects/ISMIP6Hackathon/GroupBMBtodSLR/data_analysis/tables/linear_dslc_sensitivity_from_BMB_max.csv')
df_dslcs_n = pd.read_csv('/Users/fmcc0003/issm/projects/ISMIP6Hackathon/GroupBMBtodSLR/data_analysis/tables/linear_dslc_sensitivity_from_BMB_max_intercept.csv')
df_dslcs['WestAntarctica'] = df_dslcs['region_1']
df_dslcs['EastAntarctica'] = df_dslcs['region_2']
df_dslcs['Peninsula'] = df_dslcs['region_3']
df_dslcs_n['WestAntarctica'] = df_dslcs_n['region_1']
df_dslcs_n['EastAntarctica'] = df_dslcs_n['region_2']
df_dslcs_n['Peninsula'] = df_dslcs_n['region_3']

df_time['WestAntarctica'] = df_time['region_1']
df_time['EastAntarctica'] = df_time['region_2']
df_time['Peninsula'] = df_time['region_3']

In [9]:
fpath='/Volumes/FMac/computing/data/antarctica/ismip6/hackathon/figures/linear_dslcs_until_max_dSLC/'
fname='linear_dslcs'
pth = pth_helene
pth3 = pth_ismip6hack

varea = 'iareafl'
vmelt = 'shelfmelt'
vdslc = 'dslc_anom'

factor = -1* 365.25 * 24 * 3600/900

In [16]:
for key in dic_area_of_interest.keys():
    lst = dic_area_of_interest[key]

    a4_width = 8.27  # inches
    a4_height = 11.69
    colors = ['#1f77b4',
    '#2ca02c',
    '#9467bd',
    '#e377c2']
    fs = 8

    df = df_org[df_org['Experiment']== exp].reset_index(drop=True)
    for j,exp in enumerate(expnames):
        f,ax = plt.subplots(df.shape[0]//2,2,figsize=(a4_width, a4_height))
        
        df = df_org[df_org['Experiment']== exp].reset_index(drop=True)
        dslcs_exp = df_dslcs[df_dslcs['Experiment']== exp].reset_index(drop=True)
        dslcs_n_exp =df_dslcs_n[df_dslcs_n['Experiment']== exp].reset_index(drop=True)
        df_time_exp = df_time[df_time['Experiment']== exp].reset_index(drop = True)

        for i in range(df.shape[0]):
            ix = i // 2
            iy = i % 2
            
            pth_way = df.iloc[i].Path
            
            #get bmb
            pth_bmb = f'{pth}/{df.iloc[i].Experiment}/{vmelt}/computed_{vmelt}_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
            d3 = xr.open_dataset(pth_bmb,decode_times=False )
        
            #get shelf area
            pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/{varea}/computed_{varea}_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
            d2 = xr.open_dataset(pth_area,decode_times=False )

            #get dslc
            pth_dslc = f'{pth3}/{df.iloc[i].Experiment}/{vdslc}/computed_{vdslc}_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'         
            d4 = xr.open_dataset(pth_dslc,decode_times=False )

            t_max2 = df_time_exp.iloc[i][key]
            t_max3 = 2298            
            t_max  = np.min([ t_max2 , t_max3 ])   
            tmin = np.where( d.time.values>=t_max)[0][0]
       
            if tmin <10:
                print(df.iloc[i].Model)
                pass
            else:
                #get cumulative melting and dslc
                x = np.cumsum((d3[vmelt].values[:tmin].copy() * factor/d2[varea].values[:tmin].copy()), axis=0) #cumulative melt
                y = np.cumsum((-1*d4[vdslc].values[:tmin].copy()), axis = 0) # gives cumulative dSLC
                
                dSLC_even = np.linspace(np.nanmin(x),np.nanmax(x),x.shape[0])
                dslcs = dslcs_exp.iloc[i][key]
                dslcs_n = dslcs_n_exp.iloc[i][key]
                
                ax2 = ax[ix,iy]
                ax2.scatter(x, y, color = colors[j],label = f'{exp}',marker ='.')
                ax2.plot(dSLC_even, dslcs*dSLC_even+dslcs_n, color = 'red')
                
                df_info_model = df_info[df_info['fileID']==df.iloc[i].Model ]
                df_info_model_exp = df_info_model[df_info_model['Experiment']==exp]
                meltparam =  df_info_model_exp['Melt parameterisation'].values[0]
                icefront =  df_info_model_exp['ice front'].values[0]
    
            ax2.text(0.02, 0.98,f' Model {df.iloc[i].Model}, until {t_max} \n {meltparam}, {icefront}',transform=ax2.transAxes,fontsize = fs, va='top')
            ax2.text(0.02, 0.5,f'{np.round(dslcs,1)}*dSLC+{np.round(dslcs_n,1)},',transform=ax2.transAxes,fontsize = fs, va='top',color = 'red')

        handles, labels = ax[0, 0].get_legend_handles_labels()
        f.legend(handles, labels, loc='upper right', title='Experiments')
     
        f.suptitle(f'{fname} {key} dSLC (GT/yr)',fontsize = 12)
        f.savefig(f'{fpath}{fname}_{key}_{exp}.pdf')
        plt.close()

LSCE_GRISLI2
UCSD_ISSM
LSCE_GRISLI2
UCSD_ISSM
LSCE_GRISLI2
NCAR_CISM1
NORCE_CISM5-MAR364-ERA-t1-local
NORCE_CISM5-MAR364-ERA-t1-nonlocal
UCSD_ISSM


In [22]:
dslcs_exp.iloc[i][key]

447.2907144761961